In [1]:
import os, csv, uuid, json, random, re
from datetime import datetime, timedelta
from faker import Faker
from tqdm import tqdm

SEED = 42
random.seed(SEED)
fake = Faker()
fake.seed_instance(SEED)

out_dir = "./Astra_dataset_psv"
os.makedirs(out_dir, exist_ok=True)

def iso(dt): return dt.strftime("%Y-%m-%dT%H:%M:%S")
def money(x): return f"{x:.2f}"
def oneline(s): return re.sub(r"\s+", " ", s).strip()
def minijson(d): return json.dumps(d, separators=(",",":"))
def set_literal(items): 
    return "{"+",".join(f"\"{str(x)}\"" for x in items if str(x).strip())+"}"

print("Seed:", SEED, "Output:", out_dir)

Seed: 42 Output: ./Astra_dataset_psv


In [2]:
cats = [
 'Electronics','Clothing','Home & Garden','Sports & Outdoors','Books',
 'Beauty & Personal Care','Toys & Games','Automotive','Health & Wellness','Food & Beverages',
 'Jewelry & Accessories','Office Supplies','Pet Supplies','Baby & Kids','Music & Movies',
 'Furniture','Kitchen & Dining','Tools & Hardware','Travel & Luggage','Craft & Hobby',
 'Shoes','Bags & Handbags','Watches','Sunglasses','Phone Accessories',
 'Computer Accessories','Gaming','Fitness Equipment','Outdoor Gear','Art Supplies',
 'Party Supplies','Gift Cards','Magazines','Software','Musical Instruments',
 'Collectibles','Industrial & Scientific','Religious & Spiritual','Adult','Vintage',
 'Handmade','Local Products','Organic','Eco-Friendly','Premium Brands',
 'Discount','Clearance','New Arrivals','Best Sellers','Seasonal'
]
path = f"{out_dir}/categories.psv"
with open(path,"w",newline="",encoding="utf-8") as f:
    w=csv.writer(f,delimiter="|")
    w.writerow(["id","name","name_lc","slug","parent_id","created_at","updated_at"])
    now = datetime.now()
    for i,name in enumerate(cats):
        cid = str(uuid.uuid4())
        parent = None if i<10 else str(uuid.uuid4())
        w.writerow([
            cid, name, name.lower(), name.lower().replace(" & ","-").replace(" ","-"),
            parent or "", iso(now - timedelta(days=random.randint(30,365))),
            iso(now - timedelta(days=random.randint(1,30)))
        ])
print("Wrote:", path)

Wrote: ./Astra_dataset_psv/categories.psv


In [3]:
path = f"{out_dir}/customers.psv"
with open(path,"w",newline="",encoding="utf-8") as f:
    w=csv.writer(f,delimiter="|")
    w.writerow(["id","first_name","last_name","email","email_lc","phone",
                "address_line1","address_line2","city","state","country","postal_code",
                "is_active","created_at","updated_at"])
    now = datetime.now()
    for _ in tqdm(range(25_000), desc="Customers"):
        fn, ln = fake.first_name(), fake.last_name()
        email = f"{fn.lower()}.{ln.lower()}@{fake.free_email_domain()}"
        w.writerow([
            str(uuid.uuid4()), fn, ln, email, email.lower(), fake.phone_number()[:15],
            fake.street_address(), (fake.secondary_address() if random.random()<0.3 else ""),
            fake.city(), fake.state(), "USA", fake.zipcode(),
            random.choice([True,True,True,False]),
            iso(now - timedelta(days=random.randint(1,730))),
            iso(now - timedelta(days=random.randint(1,30)))
        ])
print("Wrote:", path)

Customers: 100%|███████████████████████████████████████████████████████████████| 25000/25000 [00:13<00:00, 1792.90it/s]

Wrote: ./Astra_dataset_psv/customers.psv


In [4]:
brands = ['Apple','Samsung','Nike','Adidas','Sony','LG','Dell','HP','Canon','Nikon',
          "Levi's",'Gap','H&M','Zara','Ikea','Amazon Basics','Target','Walmart']
cat_ids = [row.split("|",1)[0] for row in open(f"{out_dir}/categories.psv",encoding="utf-8").read().splitlines()[1:]]

path = f"{out_dir}/products.psv"
with open(path,"w",newline="",encoding="utf-8") as f:
    w=csv.writer(f,delimiter="|")
    w.writerow(["id","sku","sku_lc","name","name_lc","description","category_id",
                "brand","brand_lc","price","currency","stock","popularity",
                "attributes","tags","is_active","created_at","updated_at"])
    now = datetime.now()
    for i in tqdm(range(12_000), desc="Products"):
        name = fake.catch_phrase()
        brand = random.choice(brands)
        attrs = {"color":random.choice(['Red','Blue','Green','Black','White','Gray']),
                 "size":random.choice(['XS','S','M','L','XL','XXL']) if random.random()<0.6 else None,
                 "material":random.choice(['Cotton','Polyester','Metal','Plastic','Wood']) if random.random()<0.4 else None}
        tags = random.sample(['popular','new','sale','premium','eco-friendly','bestseller'], 
                             random.randint(0,3))
        w.writerow([
            str(uuid.uuid4()), f"SKU-{i+1:06d}", f"sku-{i+1:06d}",
            name, name.lower(), oneline(fake.text(400)),
            random.choice(cat_ids),
            brand, brand.lower(), money(random.uniform(9.99,999.99)), "USD",
            random.randint(0,1000), random.randint(1,10000),
            minijson(attrs), set_literal(tags),
            random.choice([True,True,True,False]),
            iso(now - timedelta(days=random.randint(1,365))),
            iso(now - timedelta(days=random.randint(1,30)))
        ])
print("Wrote:", path)

Products: 100%|████████████████████████████████████████████████████████████████| 12000/12000 [00:06<00:00, 1838.49it/s]

Wrote: ./Astra_dataset_psv/products.psv


In [5]:
cust_ids = [row.split("|",1)[0] for row in open(f"{out_dir}/customers.psv",encoding="utf-8").read().splitlines()[1:]]
order_ids = []

path = f"{out_dir}/orders.psv"
with open(path,"w",newline="",encoding="utf-8") as f:
    w=csv.writer(f,delimiter="|")
    w.writerow(["id","customer_id","order_date","status","payment_status","currency",
                "subtotal_amount","tax_amount","shipping_fee","discount_amount",
                "total_amount","coupon_code","shipping_address","billing_address",
                "created_at","updated_at"])
    now = datetime.now()
    statuses = ['pending','processing','shipped','delivered','cancelled']
    pays = ['pending','completed','failed','refunded']
    for _ in tqdm(range(250_000), desc="Orders"):
        oid = str(uuid.uuid4()); order_ids.append(oid)
        subtotal = random.uniform(20.0,500.0)
        tax = subtotal*0.08
        ship = random.uniform(5.99,15.99)
        disc = (subtotal*random.uniform(0,0.15)) if random.random()<0.3 else 0.0
        addr1 = {"street":fake.street_address(),"city":fake.city(),"state":fake.state(),"zip":fake.zipcode()}
        addr2 = {"street":fake.street_address(),"city":fake.city(),"state":fake.state(),"zip":fake.zipcode()}
        t = now - timedelta(days=random.randint(1,365))
        w.writerow([
            oid, random.choice(cust_ids), iso(t),
            random.choice(statuses), random.choice(pays), "USD",
            money(subtotal), money(tax), money(ship), money(disc),
            money(subtotal+tax+ship-disc),
            (f"SAVE{random.randint(10,50)}" if random.random()<0.2 else ""),
            minijson(addr1), minijson(addr2),
            iso(t), iso(now - timedelta(days=random.randint(1,30)))
        ])
print("Wrote:", path, "Order IDs:", len(order_ids))

Orders: 100%|████████████████████████████████████████████████████████████████| 250000/250000 [02:38<00:00, 1580.04it/s]

Wrote: ./Astra_dataset_psv/orders.psv Order IDs: 250000


In [6]:
# Collect product IDs (only the first column of each line) efficiently
prod_ids = [row.split("|",1)[0] for row in open(f"{out_dir}/products.psv",encoding="utf-8").read().splitlines()[1:]]

TARGET = 500_000
items_per = TARGET // len(order_ids)
extra = TARGET % len(order_ids)

path = f"{out_dir}/order_items.psv"
with open(path,"w",newline="",encoding="utf-8") as f:
    w=csv.writer(f,delimiter="|")
    w.writerow(["id","order_id","product_id","quantity","unit_price","discount_amount","total_price","created_at"])
    now = datetime.now()
    pbar = tqdm(total=TARGET, desc="OrderItems")
    written = 0
    for i, oid in enumerate(order_ids):
        n = items_per + (1 if i<extra else 0)
        for _ in range(n):
            pid = random.choice(prod_ids)
            qty = random.randint(1,3)
            unit = random.uniform(5.0,500.0)
            disc = (unit*qty*random.uniform(0,0.1)) if random.random()<0.2 else 0.0
            w.writerow([str(uuid.uuid4()), oid, pid, qty, money(unit), money(disc), money(unit*qty-disc), iso(datetime.now())])
        written += n
        pbar.update(n)
    pbar.close()
print("Wrote:", path, "Rows:", written)

OrderItems: 100%|███████████████████████████████████████████████████████████| 500000/500000 [00:14<00:00, 35317.67it/s]

Wrote: ./Astra_dataset_psv/order_items.psv Rows: 500000


In [7]:
import pandas as pd, json, ast, os
from tqdm import tqdm

src = "C:/Users/avyaa/Astra_dataset_psv/products.psv"
dst = "C:/Users/avyaa/Astra_dataset_psv/products_json.psv"

def to_json_array(s):
    if pd.isna(s) or s.strip()=="":
        return "[]"
    txt = s.strip()
    if txt.startswith("{") and txt.endswith("}"):
        inner = txt[1:-1].strip()
        parts = [p.strip() for p in inner.split(",") if p.strip()!=""]
        vals = []
        for p in parts:
            if p.startswith('"') and p.endswith('"'): vals.append(p[1:-1])
            elif p.startswith("'") and p.endswith("'"): vals.append(p[1:-1])
            else: vals.append(p)
        return json.dumps(vals, separators=(",",":"))
    try:
        v = ast.literal_eval(txt)
        if isinstance(v, (list, tuple, set)):
            return json.dumps(list(v), separators=(",",":"))
    except Exception:
        pass
    return json.dumps([txt], separators=(",",":"))

chunks = pd.read_csv(src, sep="|", dtype=str, keep_default_na=False, chunksize=100_000)
header = True
for ch in tqdm(chunks, desc="Fix tags→JSON"):
    if "tags" in ch:
        ch["tags"] = ch["tags"].map(to_json_array)
    ch.to_csv(dst, sep="|", index=False, mode="w" if header else "a", header=header)
    header = False

print("Wrote:", dst)

Fix tags→JSON: 1it [00:00,  4.46it/s]

Wrote: C:/Users/avyaa/Astra_dataset_psv/products_json.psv
